# PT-Mark on Google Colab

Run this notebook after placing your `pt-mark` project folder in Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

!git clone https://github.com/ademladhari/pt-markown
!dir
%cd pt-markown

fatal: destination path 'pt-markown' already exists and is not an empty directory.
erer  pt-markown  sample_data
/content/pt-markown


In [ ]:
# Edit this path if your folder location differs
PROJECT_DIR = '/content/drive/MyDrive/pt-mark'

import os
assert os.path.exists(PROJECT_DIR), f'Project folder not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

In [ ]:
!nvidia-smi

In [2]:
# Optional: create and use venv inside Colab runtime
# Uncomment if you explicitly want venv behavior in Colab.
# !python -m venv .venv
# !. .venv/bin/activate && python -V
!dir
%cd pt-markown
# Install requirements in the Colab runtime environment
!python -m pip install --upgrade pip
!pip install --no-cache-dir -r requirements.txt

PTMark_Colab.ipynb  pt-mark.txt  requirements.txt  src
pt-markown	    README.md	 scripts
/content/erer/pt-markown


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Change prompt/seed as needed
PROMPT = "A fantasy innkeeper portrait"
SEED = 42
OUTDIR = "runs/single_colab"

# Paper defaults from PT-Mark: 50 diffusion steps and 10 null-text optimization steps.
!python scripts/run_single_ptmark.py --prompt "$PROMPT" --seed $SEED --outdir "$OUTDIR" --num-inference-steps 50 --null-opt-steps 10

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading pipeline components...:  17% 1/6 [00:00<00:03,  1.56it/s]
Loading weights:   0% 0/372 [00:00<?, ?it/s]
Loading weights:   0% 1/372 [00:00<00:00, 12300.01it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   0% 1/372 [00:00<00:00, 3412.78it/s, Materializing param=text_model.embeddings.position_embedding.weight] 
Loading weights:   1% 2/372 [00:00<00:00, 3289.65it/s, Materializing param=text_model.embeddings.token_embedding.weight]   
Loading weights:   1% 2/372 [00:00<00:00, 2275.80it/s, Materializing param=text_model.embeddings.token_embedding.weight]
Loading weights:   1% 3/372 [00:00<00:00, 2686.36it/s, Materializing param=text_model.encoder.la

In [ ]:
!python scripts/verify_single.py --prompt "$PROMPT" --image "$OUTDIR/pt_mark.png"

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading pipeline components...:  67% 4/6 [00:00<00:00,  5.92it/s]
Loading weights:   0% 0/372 [00:00<?, ?it/s]
Loading weights:   0% 1/372 [00:00<00:00, 11125.47it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   0% 1/372 [00:00<00:00, 5275.85it/s, Materializing param=text_model.embeddings.position_embedding.weight] 
Loading weights:   1% 2/372 [00:00<00:00, 4380.47it/s, Materializing param=text_model.embeddings.token_embedding.weight]   
Loading weights:   1% 2/372 [00:00<00:00, 2849.39it/s, Materializing param=text_model.embeddings.token_embedding.weight]
Loading weights:   1% 3/372 [00:00<00:00, 3264.90it/s, Materializing param=text_model.encoder.la

In [ ]:
from IPython.display import Image, display

display(Image(filename=f"{OUTDIR}/clean.png"))
display(Image(filename=f"{OUTDIR}/tree_ring_baseline.png"))
display(Image(filename=f"{OUTDIR}/pt_mark.png"))

In [ ]:
from PIL import Image
from torchvision.transforms import ToTensor

from src.evaluation.metrics import basic_image_metrics
from src.inversion.ddim import DDIMInverter
from src.pipelines.base_diffusion import DiffusionCore
from src.verification.detector import WatermarkDetector
from src.watermark.tree_ring import TreeRingWatermark
from src.config import ModelConfig, WatermarkConfig

# Evaluation command for the notebook: compute PSNR/SSIM and watermark scores.
core = DiffusionCore(ModelConfig())
inverter = DDIMInverter(core)
watermark = TreeRingWatermark(WatermarkConfig())
detector = WatermarkDetector(watermark)

clean = ToTensor()(Image.open(f"{OUTDIR}/clean.png").convert('RGB')).unsqueeze(0).to(core.cfg.device)
pt_mark = ToTensor()(Image.open(f"{OUTDIR}/pt_mark.png").convert('RGB')).unsqueeze(0).to(core.cfg.device)
baseline = ToTensor()(Image.open(f"{OUTDIR}/tree_ring_baseline.png").convert('RGB')).unsqueeze(0).to(core.cfg.device)

print('PSNR/SSIM (clean vs pt_mark):', basic_image_metrics(clean, pt_mark))
print('PSNR/SSIM (clean vs tree_ring_baseline):', basic_image_metrics(clean, baseline))

for name, path in [('clean', f"{OUTDIR}/clean.png"), ('tree_ring_baseline', f"{OUTDIR}/tree_ring_baseline.png"), ('pt_mark', f"{OUTDIR}/pt_mark.png")]:
    image = ToTensor()(Image.open(path).convert('RGB')).unsqueeze(0).to(core.cfg.device, dtype=core.pipe.unet.dtype)
    z_star = inverter.invert(PROMPT, image)
    payload = watermark.build_payload(z_star[0])
    score = detector.score(z_star[0], payload).mean().item()
    print(f'{name} watermark score: {score:.6f}')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--sd2-community--stable-diffusion-2-1-base/snapshots/4e63672c03103b6c636b8fb4119ba982469b2955/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PSNR/SSIM (clean vs pt_mark): {'psnr': 11.338450164384753, 'ssim': 0.30372190475463867}
PSNR/SSIM (clean vs tree_ring_baseline): {'psnr': 11.928918522299792, 'ssim': 0.1745729297399521}


/content/pt-markown/src/watermark/tree_ring.py:55: UserWarning: ComplexHalf support is experimental and many operators don't support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  freq = torch.fft.fft2(z_t[:, c, :, :])


clean watermark score: 22.288038
tree_ring_baseline watermark score: 8.659397
pt_mark watermark score: 4.736375
